In [23]:
from minisom import MiniSom
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pyarrow as pa
import seaborn as sns

In [2]:
dataset = fetch_openml(data_id=41960, as_frame=True)

In [3]:
dataset.DESCR

'**Author**: City of Seattle\n**Source**: https://data.seattle.gov/Public-Safety/Crime-Data/4fs7-3vj5 - 24-06-2019\n**Please cite**:   \n\nThis data represents crime reported to the Seattle Police Department (SPD). Each row contains the record of a unique event where at least one criminal offense was reported by a member of the community or detected by an officer in the field. This data is the same data used in meetings such as SeaStat (https://www.seattle.gov/police/information-and-data/seastat) for strategic planning, accountability and performance management. \n\nFor more information see:\nhttps://data.seattle.gov/Public-Safety/Crime-Data/4fs7-3vj5\n\nDownloaded from openml.org.'

In [4]:
dataset.frame.head()

,Occurred_Time,Reported_Time,Crime_Subcategory,Primary_Offense_Description,Precinct,Sector,Beat,Neighborhood
0,900.0,1500.0,BURGLARY-RESIDENTIAL,BURGLARY-FORCE-RES,SOUTH,R,R3,LAKEWOOD/SEWARD PARK
1,1.0,2359.0,SEX OFFENSE-OTHER,SEXOFF-INDECENT LIBERTIES,UNKNOWN,NaN,NaN,UNKNOWN
2,1600.0,1430.0,CAR PROWL,THEFT-CARPROWL,EAST,G,G2,CENTRAL AREA/SQUIRE PARK
3,2029.0,2030.0,HOMICIDE,HOMICIDE-PREMEDITATED-WEAPON,SOUTH,S,S2,BRIGHTON/DUNLAP
4,2000.0,435.0,BURGLARY-RESIDENTIAL,BURGLARY-FORCE-RES,SOUTHWEST,W,W3,ROXHILL/WESTWOOD/ARBOR HEIGHTS


In [5]:
df = pl.DataFrame(dataset.frame)

In [7]:
df.describe()

statistic,Occurred_Time,Reported_Time,Crime_Subcategory,Primary_Offense_Description,Precinct,Sector,Beat,Neighborhood
str,f64,f64,str,str,str,str,str,str
"""count""",523588.0,523588.0,"""523328""","""523590""","""523584""","""520244""","""520292""","""523590"""
"""null_count""",2.0,2.0,"""262""","""0""","""6""","""3346""","""3298""","""0"""
"""mean""",1358.650429,1353.362726,null,null,null,null,null,null
"""std""",688.348689,589.368521,null,null,null,null,null,null
"""min""",0.0,0.0,null,null,null,null,null,null
"""25%""",900.0,950.0,null,null,null,null,null,null
"""50%""",1500.0,1407.0,null,null,null,null,null,null
"""75%""",1920.0,1817.0,null,null,null,null,null,null
"""max""",2359.0,2359.0,null,null,null,null,null,null


In [9]:
3346/523590 * 100 

0.6390496380755936

In [10]:
3298/523590 * 100 

0.6298821597051127

como a quantidade de registros nulos nas colunas Sector, Beat, Precint, Crime_Subcategory é inferior a 5% dos registros totais, optei por dropar

In [ ]:
df_without_nulls = df.select().drop_nulls()

In [12]:
df_without_nulls.shape

(519975, 8)

In [17]:
df_without_nulls.select([pl.col(c).n_unique().alias(c) for c in df.columns])

Occurred_Time,Reported_Time,Crime_Subcategory,Primary_Offense_Description,Precinct,Sector,Beat,Neighborhood
u32,u32,u32,u32,u32,u32,u32,u32
1440,1440,30,142,5,17,51,59


como a coluna "Primary_Offense_Description" é uma versão mais detalhada da coluna "Crime_Subcategory", optei por dropar ela.

In [20]:
df_without_nulls = df_without_nulls.select("Occurred_Time", 
    "Reported_Time",
    "Crime_Subcategory",
    "Precinct", 
    "Sector",
    "Beat", 
    "Neighborhood")

In [21]:
df_without_nulls.select("Crime_Subcategory").unique()

Crime_Subcategory
cat
"""HOMICIDE"""
"""ARSON"""
"""NARCOTIC"""
"""ROBBERY-COMMERCIAL"""
"""BURGLARY-COMMERCIAL-SECURE PAR…"
…
"""TRESPASS"""
"""LIQUOR LAW VIOLATION"""
"""ROBBERY-RESIDENTIAL"""


In [22]:
df.select("Occurred_Time", "Reported_Time")

Occurred_Time,Reported_Time
f64,f64
900.0,1500.0
1.0,2359.0
1600.0,1430.0
2029.0,2030.0
2000.0,435.0
…,…
1713.0,1713.0
730.0,1721.0
1724.0,1724.0


Usando o label encoder para transformar as variáveis categórias em variáveis numéricas para a rede poder aceitar as entradas

In [24]:
label_encoder = LabelEncoder()

In [25]:
category_cols = [
    "Crime_Subcategory",
    "Precinct", 
    "Sector",
    "Beat", 
    "Neighborhood"
]

In [36]:
for col in category_cols:
    df_without_nulls = df_without_nulls.with_columns(
        pl.Series(
            col, label_encoder.fit_transform(
                df_without_nulls[col].fill_null("UNKNOWN").to_list()
            )
        )
    )

In [37]:
print(df_without_nulls.head())

shape: (5, 7)
┌───────────────┬───────────────┬───────────────────┬──────────┬────────┬──────┬──────────────┐
│ Occurred_Time ┆ Reported_Time ┆ Crime_Subcategory ┆ Precinct ┆ Sector ┆ Beat ┆ Neighborhood │
│ ---           ┆ ---           ┆ ---               ┆ ---      ┆ ---    ┆ ---  ┆ ---          │
│ f64           ┆ f64           ┆ i64               ┆ i64      ┆ i64    ┆ i64  ┆ i64          │
╞═══════════════╪═══════════════╪═══════════════════╪══════════╪════════╪══════╪══════════════╡
│ 900.0         ┆ 1500.0        ┆ 5                 ┆ 2        ┆ 13     ┆ 41   ┆ 28           │
│ 1600.0        ┆ 1430.0        ┆ 7                 ┆ 0        ┆ 5      ┆ 16   ┆ 8            │
│ 2029.0        ┆ 2030.0        ┆ 12                ┆ 2        ┆ 14     ┆ 43   ┆ 6            │
│ 2000.0        ┆ 435.0         ┆ 5                 ┆ 3        ┆ 16     ┆ 50   ┆ 49           │
│ 155.0         ┆ 155.0         ┆ 15                ┆ 4        ┆ 9      ┆ 28   ┆ 51           │
└───────────────┴─────────

Como algumas colunas tem mais de 10 valores distintos, optei por normalizar os dados pra evitar algum tipo de enviesamento causado pela magnitude dos valores

In [30]:
scaler = MinMaxScaler()

In [39]:
X = scaler.fit_transform(df_without_nulls.to_numpy())
X

array([[0.38151759, 0.63586265, 0.17241379, ..., 0.8125    , 0.82      ,
        0.48275862],
       [0.6782535 , 0.60618906, 0.24137931, ..., 0.3125    , 0.32      ,
        0.13793103],
       [0.86011022, 0.86053412, 0.4137931 , ..., 0.875     , 0.86      ,
        0.10344828],
       ...,
       [0.73081814, 0.73081814, 0.68965517, ..., 0.875     , 0.86      ,
        0.79310345],
       [0.74183976, 0.80712166, 0.93103448, ..., 0.5       , 0.5       ,
        0.70689655],
       [0.76303518, 0.94828317, 0.82758621, ..., 0.625     , 0.6       ,
        0.0862069 ]], shape=(519975, 7))

não sei se a coluna de tempo vai dar problema. anotando aqui pra revisar isso depois.

In [40]:
X.shape

(519975, 7)

testando a malha

In [41]:
som = MiniSom(30, 30, 7)

In [43]:
som.train_batch(X, num_iteration=1000000)

In [44]:
som.quantization_error(X)

MemoryError: Unable to allocate 3.49 GiB for an array with shape (519975, 900) and data type float64